# Predicting 30-Day Hospital Readmission in Patients with Diabetes

**Topic area:** Healthcare — predict patient readmission <br>
**Dataset:** Diabetes 130-US Hospitals for Years 1999–2008 (UCI: **https://archive.ics.uci.edu/dataset/296/diabetes+130+us+hospitals+for+years+1999+2008**) <br>
**Author:** Edzel Dela Cruz <br>
**Course / program:** AIM Diploma Course in AI and Machine Learning <br>

---

## 1. Problem statement


Hospitals aim to reduce the number of patients who return shortly after being discharged. Early hospital readmissions can increase healthcare costs, put additional pressure on hospital resources, and suggest that some patients may need more support after leaving the hospital.

This project uses a dataset of inpatient hospital encounters involving patients with diabetes across 130 US hospitals between 1999 and 2008. Although diabetes was recorded as a diagnosis, it was not necessarily the primary reason for hospitalization. Some patients were admitted for other conditions, such as cardiovascular or respiratory diseases. A portion of these patients were readmitted within 30 days of discharge.

Hospitals have limited resources and may not be able to provide intensive follow-up care, such as phone calls, early appointments, or home visits, to every discharged patient. The challenge is identifying which patients are at greater risk of readmission so that hospitals can prioritize those who may need additional attention and support.

    Research Question: Can we use patient information available at the time of discharge to predict the likelihood of hospital readmission within 30 days and help hospitals prioritize patients for follow-up care?


This project focuses on predicting whether a patient will be readmitted to the hospital within 30 days of discharge. The 30-day window follows the methodology of Strack et al. (2014), who selected this threshold based on criteria commonly used by funding agencies. The goal is to develop a machine learning model that can identify patients at higher risk of early readmission and provide hospitals with information to support more targeted follow-up care.

## 2. Business / organizational context

| Item | Description |
|---|---|
| Setting | US hospitals and integrated delivery networks |
| Data source | Retrospective extract from the Health Facts database: 10 years (1999–2008), 130 hospitals (Strack et al., 2014) |
| Decision this supports | Which discharged patients receive extra follow-up |
| Decision makers | Discharge planners, care-management teams, hospital quality leaders |
| Nature of this project | A review of historical record for a capstone. It is **not** a deployed clinical tool and must not be used to make real patient decisions |

## 3. Business objective

Help a hospital focus limited follow-up resources on the patients most likely to be readmitted within 30 days, so that more early readmissions can be reached with the same amount of effort.

## 4. Machine learning objective

Build and compare supervised classification models that estimate each patient's probability of 30-day readmission from encounter information (demographics, admission and discharge details, diagnoses, lab tests, medications, prior visits). Select the best model using evidence from a validation set, explain its predictions, and audit it for overfitting, leakage and fairness.

## 5. Target variable

| Item | Definition / final dataset | Raw dataset |
|---|---|---|
| Name | `readmit_30` | Original column: `readmitted` |
| Source column | `readmitted` (original values: `<30`, `>30`, `NO`) | Same three original labels |
| Value 1 | Readmitted within 30 days (`readmitted == "<30"`) | `<30`: **11,357 encounters (11.16%)** |
| Value 0 | Readmitted after 30 days (`>30`) or no recorded readmission (`NO`) | `>30` + `NO`: **90,409 encounters (88.84%)** |
| Class balance | **6,277 positives (8.97%)** and **63,693 negatives (91.03%)** out of 69,970 patients | **11,357 positives (11.16%)** and **90,409 negatives (88.84%)** out of 101,766 encounters |

The original `readmitted` column is removed from the features after the target is created, so it cannot leak into the model.

## 6. Machine learning task type

**Supervised binary classification** with **imbalanced classes** (about 1 positive for every 10 negatives). Because of the imbalance, plain accuracy is not used: a model that predicts "not readmitted" for everyone would be about 91% accurate and would find no readmissions at all.

## 7. Potential benefits

| Group | Potential benefit |
|---|---|
| Care managers and discharge planners | A ranked list of patients to prioritise for follow-up |
| Hospital leadership and quality teams | Better use of limited resources, and a way to monitor readmission performance |
| Patients | Earlier support for those at highest risk |
| Clinicians | Insight into which factors are associated with early readmission |

These are expected benefits if a suitable follow-up program is implemented. This capstone measures prediction and prioritization. It does not establish that follow-up prevents readmissions, reduces costs, or improves patient outcomes.

## 8. Success criteria

### 8.1 Technical success metrics

Model and policy selection use **training and validation data**. Final metrics and diagnostic audits evaluate the **fixed model on the test set**. Test outcomes are not used to choose new settings. The completed results and qualifications are reported in notebook 6, section 6.11.

| Metric | Role | Plain-language meaning |
|---|---|---|
| **Average precision (AP)** | **Primary** | How well the model finds true readmissions while keeping its alerts accurate. A random model scores about 0.0897 (the share of positives) |
| **ROC-AUC** | Secondary | The chance that the model ranks a randomly chosen readmitted patient above a randomly chosen non-readmitted one. 0.50 means no better than a coin flip |
| **Recall and precision** at a chosen threshold | Secondary | Of all actual readmissions, how many we catch (recall), and of all patients we flag, how many are truly readmitted (precision) |

The code uses `average_precision_score`. AP summarizes precision–recall performance and can differ from trapezoidal area under a precision–recall curve. I use AP consistently as the primary ranking metric.

### 8.2 Business KPI (measurable)

**Readmission capture rate at the top 20%:** the share of all 30-day readmissions found among the 20% of patients the model ranks as highest risk.

- A random ranking captures about **20%** (by definition).
- **Lift** = capture rate ÷ the actual fraction of patients selected (approximately 20%). For example, a capture rate of 30% is a lift of 1.5.
- This translates directly into follow-up workload: "If we contact the top 20% of discharged patients, we reach X% of the patients who would have been readmitted."

### 8.3 Success criteria

| # | Criterion | Status |
|---|---|---|
| 1 | The final model beats the dummy baseline and a simple logistic-regression baseline on validation AP and ROC-AUC | **Met in point estimates only.** Tuned boosting AP = 0.1696 and ROC-AUC = 0.6454 exceed both baselines, but paired tests do not establish superiority over full logistic regression after Holm correction. |
| 2 | Top-20% capture rate on the test set is at least **30%** (lift of at least 1.5). This remains the original project target. | **Met.** Capture = 39.17%, lift = 1.96×. Approximate bootstrap 95% capture interval = 35.93%–41.60%. |
| 3 | Validation and test results are consistent, with no sign of overfitting | **Partially met.** Held-out results are broadly similar, but training AP remains higher than validation AP. |
| 4 | No data leakage: one row per patient, target column removed from features, all preprocessing fitted on training data only | **Safeguards implemented, with qualifications.** Exact manifest checks cover all 69,970 patients with zero overlap. Medication exclusions use training frequencies only. See notebook 6, section 6.7. |
| 5 | Performance is reported by subgroup (race, gender, age group) and any large gaps are documented with proposed mitigations | **Reporting completed.** Subgroup rates, parity/error-rate metrics, uncertainty intervals and mitigation proposals are saved. Coverage gaps remain; fairness is not established. See notebook 6, section 6.8. |

The objectives above define the project. The status column now summarizes the completed outputs in notebooks 5 and 6. The original 30% capture target is retained rather than revised after seeing results. Confidence intervals describe the existing fitted model on this test population, not performance in future hospitals. Paired validation comparisons in notebook 05 use 5,000 stratified bootstrap samples for AP (approximate normal bootstrap-SE p-values), paired DeLong tests for ROC-AUC, and Holm correction across 30 tests. Tuned boosting significantly outperforms SVD logistic regression on AP (see current adjusted p-value table) and ROC-AUC (see current adjusted p-value table), but superiority over full or selected logistic regression, random forest, and original boosting is not established after correction. These are exploratory post-selection comparisons; nonsignificance does not prove equivalence.

## 9. Analysis cohort

| Step | Rows remaining |
|---|---|
| Raw dataset | 101,766 |
| Keep first encounter per patient (avoids the same patient appearing in both training and test data) | 71,518 |
| Remove discharges to death or hospice (codes 11, 13, 14, 19, 20, 21), following Strack et al. | 69,973 |
| Remove 3 records with invalid gender | **69,970** |

Train / validation / test split: 70% / 15% / 15%, stratified on the target, `random_state=42`.

## 10. Assumptions and scope limits

- **Time of prediction:** the model is meant to be used at discharge, so it uses only information recorded for the encounter and the preceding year.
- **"First" encounter:** the file has no admission dates, so "first encounter" is defined by the lowest `encounter_id`. This assumes the ID order reflects time order.
- **Readmissions outside the network are not captured:** Strack et al. state that data from out-of-network providers is not included, so some true readmissions may be labelled as not readmitted.
- **Age of the data:** the records cover 1999–2008. Clinical practice and coding have changed since, so results may not transfer to today's hospitals.
- **Diabetes encounters only:** these are encounters where diabetes was coded, not all encounters of diabetic patients.
- **Association, not causation:** the model finds patterns linked to readmission. It does not show that changing a factor (for example HbA1c testing) would prevent readmission.

## 11. References

1. Strack, B., DeShazo, J. P., Gennings, C., Olmo, J. L., Ventura, S., Cios, K. J., & Clore, J. N. (2014). Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records. *BioMed Research International*, 2014, Article ID 781670. https://doi.org/10.1155/2014/781670
2. Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). Diabetes 130-US Hospitals for Years 1999-2008 [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5230J.

